In [17]:
import numpy as np

In [18]:
def BN_forward_pass(z, a, b):
  
  caches = {}
  
  # find the mean of batch
  f1 = np.mean(z, axis=1, keepdims=True)
  caches["f1"] = f1
      
  # find the variance of the batch 
  f2 = z - f1
  caches["f2"] = f2
  
  f3 = np.power(f2, 2)
  caches["f3"] = f3
  
  f4 = np.mean(f3, axis=1, keepdims=True)
  caches["f4"] = f4
      
  # normalization
  eps = 1e-10 
  f5 = np.sqrt(f4 + eps)
  caches["f5"] = f5
  
  f6 = 1 / f5
  caches["f6"] = f6
  
  f7 = f2 * f6
  caches["f7"] = f7
  
  # scaling & shifting
  z_hat = f7 * a + b
      
  return z_hat, caches
  

In [19]:
def BN_backward_pass(caches, grad_output, gamma):
  
  # grad_output = dL/dz`
  
  # for example: if z` = [1, 2, 3], dL/dz = [0, 0.33, 3],
  #                     [1, -2, 3],         [0.2, -2, 3],
  #                     [0, 2, 10],         [0.03, 2, 10],
  
  grads = {}
  I = caches["f2"].shape[1]
  eps = 1e-10
  
  grads["dL_df7"] = grad_output * gamma
  grads["dL_dγ"] = np.sum(grad_output * caches["f7"])
  grads["dL_dδ"] = np.sum(grad_output)
  
  grads["dL_df6"] = np.sum(grads["dL_df7"] * caches["f2"])
  grads["dL_df2_direct"] = grads["dL_df7"] * caches["f6"]
  
  grads["dL_df5"] = grads["dL_df6"] * (-1 / (caches["f5"]**2))
  
  grads["dL_df4"] = grads["dL_df5"] * (1 / (2 * np.sqrt(caches["f4"] + eps)))
  
  grads["dL_df3"] = grads["dL_df4"] * (1 / I)
  
  grads["dL_df2_variance"] = grads["dL_df3"] * 2 * caches["f2"]
  grads["dL_df2_final"] = grads["dL_df2_direct"] + grads["dL_df2_variance"]
  
  grads["dL_df1"] = np.sum(grads["dL_df2_final"] * (-1))
  
  grads["dL_dz"] = grads["dL_df2_final"] + (grads["dL_df1"] * (1 / I)) 
  
  return grads

In [20]:
z = np.array(
  # features x samples
  [
    [1, 2, 3],
    [1, -2, 3],
    [0, 2, 10],
  ]
)
print("z: ", z)
print(" ")

gamma = 1
delta = 0

z_hat, caches = BN_forward_pass(z, gamma, delta)
print(f"z_hat: {z_hat}")
print(f"caches: {caches}")

dL_dz_hat = z = np.array(
  # features x samples
  [
    [0.1, 0.2, 1],
    [0.11, -0.2, 0.3],
    [0.112, 0.2, 0.10],
  ]
) 

grads = BN_backward_pass(caches, dL_dz_hat, gamma)
print(" ")
print(f"grads: {grads}")

z:  [[ 1  2  3]
 [ 1 -2  3]
 [ 0  2 10]]
 
z_hat: [[-1.22474487  0.          1.22474487]
 [ 0.16222142 -1.29777137  1.13554995]
 [-0.9258201  -0.46291005  1.38873015]]
caches: {'f1': array([[2.        ],
       [0.66666667],
       [4.        ]]), 'f2': array([[-1.        ,  0.        ,  1.        ],
       [ 0.33333333, -2.66666667,  2.33333333],
       [-4.        , -2.        ,  6.        ]]), 'f3': array([[ 1.        ,  0.        ,  1.        ],
       [ 0.11111111,  7.11111111,  5.44444444],
       [16.        ,  4.        , 36.        ]]), 'f4': array([[ 0.66666667],
       [ 4.22222222],
       [18.66666667]]), 'f5': array([[0.81649658],
       [2.05480467],
       [4.3204938 ]]), 'f6': array([[1.22474487],
       [0.48666426],
       [0.23145502]]), 'f7': array([[-1.22474487,  0.        ,  1.22474487],
       [ 0.16222142, -1.29777137,  1.13554995],
       [-0.9258201 , -0.46291005,  1.38873015]])}
 
grads: {'dL_df7': array([[ 0.1  ,  0.2  ,  1.   ],
       [ 0.11 , -0.2  ,  0.

In [21]:
np.mean(z, axis=1, keepdims=True)

array([[0.43333333],
       [0.07      ],
       [0.13733333]])